# Housing Segment Classification

## Project overview

This case study demonstrates a two-stage analytical workflow. First, **K-Means clustering** is used to create data-driven housing segments. The two largest clusters are then treated as target classes for supervised classification using **Logistic Regression** and **Linear Discriminant Analysis (LDA)**.

**Business-style question:** After identifying housing segments with clustering, can the segment assignment be reproduced using supervised classification methods?

**Methods:** K-Means, feature standardisation, train/test split, Logistic Regression, Linear Discriminant Analysis, confusion matrix, precision, recall, F1-score and accuracy.

### Important methodological limitation
The target classes are created from the same housing features later used by the classifiers. High accuracy therefore shows that the models can reproduce the K-Means partition; it should **not** be interpreted as performance against an independent real-world outcome.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.metrics import classification_report, confusion_matrix

## 1. Create data-driven segments

The original housing sample is partitioned into four clusters. The two largest clusters are retained for the classification demonstration so that the resulting two-class dataset is reasonably balanced.


In [ ]:
# Load the dataset
file_path = '../data/california_housing.csv'
data = pd.read_csv(file_path)

# Selecting features for clustering
features = data[['housing_median_age', 'total_rooms', 'total_bedrooms', 'population', 'households', 'median_income']]

# Applying KMeans clustering
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
data['class'] = kmeans.fit_predict(features)

# Display the count of observations per cluster class
cluster_counts = data['class'].value_counts()
print(cluster_counts)

class
0    143
1    128
2     22
3      7
Name: count, dtype: int64


In [ ]:
# Keep the two largest clusters used in the original analysis
df = data[~data['class'].isin([2, 3])].copy().reset_index(drop=True)

print(df['class'].value_counts())
df.head()

## 2. Data quality and class balance


In [ ]:
# Review of data frame information: dimention, list of variables/features with information about the data type
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 271 entries, 0 to 270
Data columns (total 8 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   housing_median_age  271 non-null    float64
 1   total_rooms         271 non-null    float64
 2   total_bedrooms      271 non-null    float64
 3   population          271 non-null    float64
 4   households          271 non-null    float64
 5   median_income       271 non-null    float64
 6   median_house_value  271 non-null    float64
 7   class               271 non-null    int64  
dtypes: float64(7), int64(1)
memory usage: 17.1 KB


In [ ]:
# Check the na records in dataframe
df.isna().sum()

housing_median_age    0
total_rooms           0
total_bedrooms        0
population            0
households            0
median_income         0
median_house_value    0
class                 0
dtype: int64

In [ ]:
df['class'].value_counts()

class
0    143
1    128
Name: count, dtype: int64

The retained sample contains two fairly balanced classes, which makes it suitable for a simple classification comparison.


## 3. Standardisation and train/test split

The original experiment standardised the numerical features and then created a 70/30 train/test split. This sequence is preserved here so the reported results remain traceable to the original analysis. In a production workflow, the scaler should be fitted on the training sample only and then applied to the test sample.


In [ ]:
feature_columns = [
    'housing_median_age', 'total_rooms', 'total_bedrooms', 'population',
    'households', 'median_income', 'median_house_value'
]

scaler = StandardScaler()
scaled_features = scaler.fit_transform(df[feature_columns])

df_scaled = pd.DataFrame(scaled_features, columns=feature_columns, index=df.index)
df_scaled['class'] = df['class'].values

X_train, X_test, y_train, y_test = train_test_split(
    df_scaled[feature_columns],
    df_scaled['class'],
    test_size=0.30,
    random_state=100
)

print('Training samples:', len(X_train))
print('Testing samples:', len(X_test))

## 4. Logistic Regression


In [ ]:
logistic_model = LogisticRegression(random_state=100, solver='lbfgs', max_iter=1000)
logistic_model.fit(X_train, y_train)

logistic_predictions = logistic_model.predict(X_test)
print(confusion_matrix(y_test, logistic_predictions))
print(classification_report(y_test, logistic_predictions))

array([[44,  3],
       [ 2, 33]])

              precision    recall  f1-score   support

           0       0.96      0.94      0.95        47
           1       0.92      0.94      0.93        35

    accuracy                           0.94        82
   macro avg       0.94      0.94      0.94        82
weighted avg       0.94      0.94      0.94        82



The original run reported approximately **94% accuracy** for Logistic Regression on the retained two-class sample.


## 5. Linear Discriminant Analysis


In [ ]:
lda = LDA(n_components=1, solver='svd', store_covariance=True)
lda.fit(X_train, y_train)

lda_predictions = lda.predict(X_test)
print(confusion_matrix(y_test, lda_predictions))
print(classification_report(y_test, lda_predictions))
print('Accuracy:', lda.score(X_test, y_test))

array([[45,  2],
       [ 2, 33]])

              precision    recall  f1-score   support

           0       0.96      0.96      0.96        47
           1       0.94      0.94      0.94        35

    accuracy                           0.95        82
   macro avg       0.95      0.95      0.95        82
weighted avg       0.95      0.95      0.95        82



0.9512195121951219

The original run reported an LDA accuracy of approximately **95.1%**, with similar precision and recall across the two classes.


## 6. One-dimensional discriminant space

For two classes, LDA produces one discriminant dimension. The transformed values can be inspected to see how clearly the two generated segments separate.


In [ ]:
X_train_lda = lda.transform(X_train)

plt.figure(figsize=(8, 4))
plt.hist(X_train_lda[y_train.to_numpy() == 0, 0], bins=10, alpha=0.6, label='Class 0')
plt.hist(X_train_lda[y_train.to_numpy() == 1, 0], bins=10, alpha=0.6, label='Class 1')
plt.title('Distribution in LDA Discriminant Space')
plt.xlabel('Discriminant score')
plt.ylabel('Frequency')
plt.legend()
plt.show()

## Conclusion

Both supervised models reproduced the two K-Means-derived housing segments with high accuracy in the original experiment. Logistic Regression reported about **94% accuracy**, while LDA reported about **95%**.

The exercise demonstrates how clustering and supervised classification can be combined in one analytical workflow. However, because the class labels were generated from the same input variables, the results primarily measure how well the classifiers recover the clustering structure. A real predictive application would require an independently defined target and a leakage-safe preprocessing pipeline.
